In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/santander-customer-satisfaction/sample_submission.csv
/kaggle/input/competitions/santander-customer-satisfaction/train.csv
/kaggle/input/competitions/santander-customer-satisfaction/test.csv


In [2]:
train = pd.read_csv('/kaggle/input/competitions/santander-customer-satisfaction/train.csv')

In [3]:
num_cols = [col for col in train.columns if train[col].nunique() > 2]
num_cols.pop(0)

'ID'

In [4]:
test = pd.read_csv('/kaggle/input/competitions/santander-customer-satisfaction/test.csv')

In [5]:
test

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var29_ult3,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38
0,2,2,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40532.100000
1,5,2,35,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,45486.720000
2,6,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,46993.950000
3,7,2,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,187898.610000
4,9,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,73649.730000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75813,151831,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40243.200000
75814,151832,2,26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,146961.300000
75815,151833,2,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,167299.770000
75816,151834,2,40,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.set_output(transform='pandas')

StandardScaler()

In [7]:
train[num_cols] = scaler.fit_transform(train[num_cols])

In [8]:
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0)
train_reduced = selector.fit_transform(train)

selected_features = train.columns[selector.get_support()]

In [9]:
selected_features_test = selected_features[0:336]

In [10]:
train = train[selected_features]
y = np.array(train['TARGET'])
X = np.array(train.iloc[:,1:336])

test[num_cols] = scaler.transform(test[num_cols])
test = test[selected_features_test]
X_test = np.array(test.iloc[:,1:])
test_keys = test['ID']

In [11]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

model = keras.Sequential([
    layers.Input(shape=(335,)),
    layers.Dense(32, activation='relu'),
    #layers.BatchNormalization(),
    layers.Dropout(rate=.5),
    layers.Dense(8, activation='relu'),
    layers.Dropout(rate=.4),
    #layers.BatchNormalization(),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=.00005), #1e-6 is probably best, but use like 100 epochs
        loss="binary_crossentropy",
        metrics=[keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.AUC(name="auc")]
    )

early_stop = EarlyStopping(
    monitor='val_auc', 
    patience=10, 
    restore_best_weights=True
)

class_weight = {0: 1.0, 1: 10}
model.fit(X, y, epochs=100, batch_size=50, class_weight=class_weight, validation_split=.2, callbacks=[early_stop])

2026-05-03 01:38:22.670915: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777772302.869795      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777772302.933764      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777772303.419089      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777772303.419146      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777772303.419149      16 computation_placer.cc:177] computation placer alr

Epoch 1/100
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.3390 - auc: 0.5387 - loss: 309326.9688 - val_accuracy: 0.8177 - val_auc: 0.7230 - val_loss: 165233.6094
Epoch 2/100
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6216 - auc: 0.5806 - loss: 325962.5000 - val_accuracy: 0.9065 - val_auc: 0.7475 - val_loss: 107306.4453
Epoch 3/100
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7240 - auc: 0.5967 - loss: 112750.3203 - val_accuracy: 0.9273 - val_auc: 0.7569 - val_loss: 101749.7188
Epoch 4/100
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7819 - auc: 0.6307 - loss: 164490.7344 - val_accuracy: 0.9338 - val_auc: 0.7680 - val_loss: 93758.6562
Epoch 5/100
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8164 - auc: 0.6406 - loss: 98467.0312 - val_accuracy: 0.9394 - val_auc: 0.7737 - val_loss: 90643.9141
Epoch 6/100
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8505 - auc: 0.6528 - loss: 212309.3906 - val_accuracy: 0.9386 - val_a

In [12]:
preds = model.predict(X_test)
results = pd.DataFrame({'id': test_keys, 'TARGET':preds.flatten()})
results.sort_values(by='TARGET', ascending=False)

2370/2370 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


,id,TARGET
28034,56046,0.825106
37089,74215,0.823287
40626,81235,0.787996
3452,6889,0.774599
14106,28109,0.772400
...,...,...
68906,138026,0.000000
53893,108081,0.000000
60157,120574,0.000000
41915,83737,0.000000


In [13]:
results.to_csv('submission.csv', index=False)